In [41]:
#Import The required libraries 
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
 
OUT = sys.argv[1] if len(sys.argv) > 1 else "out"
g = pd.read_csv("skill_salary_clean.csv")
ind = pd.read_csv("india_salary_clean.csv")
e = pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/economic_indicators_clean.csv')

In [42]:
#Read the files 
skill_salary_Cleaned=pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/skill_salary_clean.csv')
india_salary_Cleaned=pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/india_salary_clean.csv')
economic_indicators_Cleaned=pd.read_csv('/Users/jahaankachwala/Downloads/Projects /Global Salary Data/economic_indicators_clean.csv')

In [44]:
#1. Prepare 
g["is_india"] = (g.country_name == "India").astype(int)
g["n_skills"] = g.skills.fillna("").apply(lambda s: len(s.split("|")) if s else 0)
rank = {"intern": 0, "entry": 1, "mid": 2, "senior": 3, "lead": 4,
        "manager": 5, "director": 6, "executive": 7}
g["sen_rank"] = g.seniority.map(rank)
junior = g[g.seniority.isin(["entry", "intern"])].copy()   # our "junior" definition
 
print("=== 1. JUNIOR PAY (USD/yr) BY COUNTRY: mean, median, mode ===")
def mode_k(x):  # mode of salary rounded to nearest $1000 (salary is continuous)
    return (x / 1000).round().mode().iloc[0] * 1000
t = (junior.groupby("country_name").salary_mid_usd
     .agg(n="count", mean="mean", median="median", mode=mode_k)
     .query("n >= 20").sort_values("median", ascending=False).round(0))
print(t.to_string())
 
us_med = junior[junior.country_name == "United States"].salary_mid_usd.median()
in_med = junior[junior.is_india == 1].salary_mid_usd.median()
print(f"\nIndia junior median ${in_med:,.0f} vs US ${us_med:,.0f} -> India = {in_med/us_med:.1%} of US")

=== 1. JUNIOR PAY (USD/yr) BY COUNTRY: mean, median, mode ===
                   n     mean   median     mode
country_name                                   
Germany           23  67057.0  53753.0  54000.0
Netherlands       54  47302.0  43483.0  51000.0
United States   6358  51214.0  41600.0  42000.0
France            50  49167.0  37360.0  37000.0
Canada           870  47267.0  37143.0  30000.0
United Kingdom    20  43651.0  36184.0  36000.0
Singapore        111  29318.0  25191.0  30000.0
India           1645   4262.0   3488.0   3000.0

India junior median $3,488 vs US $41,600 -> India = 8.4% of US


In [45]:
#2. Hypothesis test  
print("\n=== 2. TEST: India junior vs everyone else junior ===")
a = junior[junior.is_india == 1].salary_mid_usd
b = junior[junior.is_india == 0].salary_mid_usd
u, p = stats.mannwhitneyu(a, b, alternative="less")   # non-parametric: salaries are skewed
print(f"Mann-Whitney U one-sided (India < rest): p = {p:.3g}")
print(f"Median India ${a.median():,.0f} | Median rest ${b.median():,.0f}")
# bootstrap CI for the median ratio
rng = np.random.default_rng(0)
ratios = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(b, len(b))) for _ in range(1000)]
print(f"India/rest median ratio 95% CI: {np.percentile(ratios,2.5):.3f} - {np.percentile(ratios,97.5):.3f}")


=== 2. TEST: India junior vs everyone else junior ===
Mann-Whitney U one-sided (India < rest): p = 0
Median India $3,488 | Median rest $41,600
India/rest median ratio 95% CI: 0.084 - 0.084


In [46]:
#3. Same job type, India vs US (controls for role)
print("\n=== 3. JUNIOR, SAME ROLE FAMILY: India vs US median USD ===")
j2 = junior[junior.country_name.isin(["India", "United States"])]
r = j2.pivot_table(index="role_family", columns="country_name",
                   values="salary_mid_usd", aggfunc=["median", "count"])
r.columns = ["India_med", "US_med", "India_n", "US_n"]
r = r[(r.India_n >= 30) & (r.US_n >= 30)]
r["India_pct_of_US"] = (r.India_med / r.US_med * 100).round(1)
print(r.sort_values("India_pct_of_US").round(0).to_string())


=== 3. JUNIOR, SAME ROLE FAMILY: India vs US median USD ===
                  India_med   US_med  India_n  US_n  India_pct_of_US
role_family                                                         
marketing            2634.0  49920.0      158   201              5.0
software-eng         4070.0  75000.0      256    39              5.0
design               3052.0  52000.0       64    33              6.0
finance              3488.0  58240.0       72   324              6.0
hr                   3198.0  44221.0       94   109              7.0
other                3140.0  43680.0      249   884              7.0
sales                3488.0  45000.0      284   761              8.0
customer-service     3198.0  40000.0      109   271              8.0
admin                3488.0  42640.0       54   185              8.0
manufacturing        3488.0  41600.0       49   671              8.0
healthcare           5233.0  45760.0       39   795             11.0


In [47]:
#4. Seniority progression
print("\n=== 4. MEDIAN USD BY SENIORITY (India vs US vs Canada) ===")
s = (g[g.country_name.isin(["India", "United States", "Canada"])]
     .pivot_table(index="seniority", columns="country_name",
                  values="salary_mid_usd", aggfunc="median")
     .reindex(list(rank)).round(0))
s["India_pct_of_US"] = (s["India"] / s["United States"] * 100).round(1)
print(s.to_string())
print("\nEntry->Senior multiple: India x%.1f, US x%.1f, Canada x%.1f" % (
    s.loc["senior", "India"] / s.loc["entry", "India"],
    s.loc["senior", "United States"] / s.loc["entry", "United States"],
    s.loc["senior", "Canada"] / s.loc["entry", "Canada"]))


=== 4. MEDIAN USD BY SENIORITY (India vs US vs Canada) ===
country_name   Canada    India  United States  India_pct_of_US
seniority                                                     
intern        44286.0   1744.0        41600.0              4.2
entry         37143.0   3779.0        41600.0              9.1
mid               NaN   7558.0        70000.0             10.8
senior        64996.0  11047.0       100000.0             11.0
lead          69998.0  13953.0        80000.0             17.4
manager       55863.0   8721.0        83232.0             10.5
director      87500.0  26163.0       130000.0             20.1
executive     85864.0  49419.0       160000.0             30.9

Entry->Senior multiple: India x2.9, US x2.4, Canada x1.7


In [48]:
#5. Correlations 
print("\n=== 5. SPEARMAN CORRELATION with salary_mid_usd ===")
for name, d in [("India", g[g.is_india == 1]),
                ("United States", g[g.country_name == "United States"])]:
    d = d.dropna(subset=["sen_rank"])
    r1 = stats.spearmanr(d.sen_rank, d.salary_mid_usd)[0]
    d2 = d[d.skills.notna()]
    r2 = stats.spearmanr(d2.n_skills, d2.salary_mid_usd)[0]
    print(f"{name}: seniority rank r={r1:.2f} (n={len(d)}) | number of skills r={r2:.2f} (n={len(d2)})")
 
# Skill premium inside India (juniors + all, USD)
ind_g = g[(g.is_india == 1) & g.skills.notna()].copy()
ind_g["skill_list"] = ind_g.skills.str.split("|")
ex = ind_g.explode("skill_list")
sk = ex.groupby("skill_list").salary_mid_usd.agg(["count", "median"])
sk = sk[sk["count"] >= 100].sort_values("median", ascending=False)
print("\nTop 8 / bottom 5 skills by median USD pay in India (n>=100 postings):")
print(pd.concat([sk.head(8), sk.tail(5)]).round(0).to_string())


=== 5. SPEARMAN CORRELATION with salary_mid_usd ===
India: seniority rank r=0.54 (n=6576) | number of skills r=0.16 (n=6538)
United States: seniority rank r=0.48 (n=24400) | number of skills r=0.23 (n=6813)

Top 8 / bottom 5 skills by median USD pay in India (n>=100 postings):
                              count   median
skill_list                                  
spring boot                     132  17442.0
azure                           141  12401.0
java                            339  12209.0
react.js                        129  12209.0
aws                             186  11482.0
management                      108  11338.0
python                          569  10174.0
sql                             467   9302.0
customer service                153   4070.0
digital marketing               136   3837.0
communication skills            259   3605.0
effective communication         123   3256.0
english proficiency (spoken)    121   2907.0


In [49]:
#6. Regression + ML (adjusted gap) 
print("\n=== 6. REGRESSION / ML: log salary ~ India + seniority + role + skills ===")
m = g[g.country_name.isin(["India", "United States", "Canada", "Singapore", "France",
                           "Germany", "Netherlands"])].dropna(subset=["sen_rank"]).copy()
m["log_sal"] = np.log(m.salary_mid_usd)
X = pd.get_dummies(m[["is_india", "sen_rank", "n_skills", "role_family"]],
                   columns=["role_family"], drop_first=True).astype(float)
y = m.log_sal
lr = LinearRegression().fit(X, y)
coef = dict(zip(X.columns, lr.coef_))
print(f"rows={len(m)}  R2={lr.score(X, y):.2f}")
print(f"India coefficient = {coef['is_india']:.2f}  -> India pays {1-np.exp(coef['is_india']):.0%} less "
      f"after holding seniority, role and skill count equal")
print(f"Each seniority step = +{np.exp(coef['sen_rank'])-1:.0%} pay | each extra skill = +{np.exp(coef['n_skills'])-1:.1%}")
 
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
rf = RandomForestRegressor(n_estimators=150, max_depth=12, n_jobs=-1, random_state=0).fit(Xtr, ytr)
print(f"Random forest test R2 = {r2_score(yte, rf.predict(Xte)):.2f}")
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top feature importances:\n", imp.head(6).round(3).to_string())


=== 6. REGRESSION / ML: log salary ~ India + seniority + role + skills ===
rows=37795  R2=0.72
India coefficient = -2.18  -> India pays 89% less after holding seniority, role and skill count equal
Each seniority step = +15% pay | each extra skill = +0.4%
Random forest test R2 = 0.76
Top feature importances:
 is_india                          0.785
sen_rank                          0.142
n_skills                          0.025
role_family_retail-hospitality    0.020
role_family_other                 0.005
role_family_software-eng          0.004


In [50]:
#7. India file: 0-1 yr experience, in INR 
print("\n=== 7. INDIA FILE: salary_mid_inr by experience band (INR/yr) ===")
e = ind.groupby("experience_band").salary_mid_inr.agg(["count", "mean", "median"]).round(0)
print(e.to_string())
print("\nFreshers (0-1 yrs) median by city (n>=50):")
f = (ind[ind.experience_band == "0-1 yrs"].groupby("city").salary_mid_inr
     .agg(["count", "median"]).query("count >= 50").sort_values("median", ascending=False).round(0))
print(f.head(8).to_string())
fresher = ind[ind.experience_band == "0-1 yrs"].salary_mid_inr.median()
print(f"\nFresher median INR {fresher:,.0f} = about USD {fresher/86:,.0f} at 86 INR/USD")


=== 7. INDIA FILE: salary_mid_inr by experience band (INR/yr) ===
                 count       mean     median
experience_band                             
0-1 yrs           3064   374610.0   300000.0
11-15 yrs          404  2960256.0  1800000.0
16+ yrs             92  3575815.0  2125000.0
2-3 yrs           3318   742169.0   525000.0
4-6 yrs           3244  1295767.0   850000.0
7-10 yrs          1743  1933586.0  1300000.0

Freshers (0-1 yrs) median by city (n>=50):
           count    median
city                      
Gurugram     147  350000.0
Noida        108  345000.0
Bengaluru    489  342000.0
Ahmedabad    213  325000.0
Mumbai       387  325000.0
Thane         61  325000.0
Delhi        199  300000.0
Hyderabad    233  300000.0

Fresher median INR 300,000 = about USD 3,488 at 86 INR/USD


In [51]:
from pathlib import Path

#8. Plots 
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
top = t.head(8)
ax[0].barh(top.index[::-1], top["median"][::-1], color="#4c72b0")
ax[0].set_title("Junior median pay (USD) by country")
ax[0].set_xlabel("USD / year")
s[["India", "United States", "Canada"]].plot(ax=ax[1], marker="o")
ax[1].set_title("Median pay by seniority (USD)")
ax[1].set_ylabel("USD / year")
junior_cmp = junior[junior.country_name.isin(["India", "United States", "Canada"])]
for c in ["India", "United States", "Canada"]:
    ax[2].hist(np.log10(junior_cmp[junior_cmp.country_name == c].salary_mid_usd),
               bins=40, alpha=0.5, label=c, density=True)
ax[2].set_title("Junior salary distribution (log10 USD)")
ax[2].legend()
plt.tight_layout()
OUT = Path("out")
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "salary_analysis.png", dpi=130)
 
fig, ax = plt.subplots(figsize=(7, 5))
imp.head(8)[::-1].plot.barh(ax=ax, color="#dd8452")
ax.set_title("What drives log salary? (random forest importance)")
plt.tight_layout()
plt.savefig(f"{OUT}/feature_importance.png", dpi=130)
print("\nSaved charts and tables to", OUT)


Saved charts and tables to out


In [52]:
#9. GDP latest year per country 
yrs = [c for c in e.columns if c.isdigit()]
# GDP latest available year per country
e = economic_indicators_Cleaned.copy()
e.columns = e.columns.map(str)  # handle year columns stored as integers

yrs = sorted((c for c in e.columns if c.isdigit()), key=int)
if not yrs:
    raise ValueError(f"No year columns found. Columns are: {e.columns.tolist()}")

print("Countries with GDP by year (non-null):",
      {year: int(e[year].notna().sum()) for year in yrs})

# For each country, use its latest non-null GDP value
e["gdp"] = e[yrs[::-1]].bfill(axis=1).iloc[:, 0]
gdp = e.set_index("country_name")["gdp"]
e["gdp"] = e["2024"].fillna(e["2023"])          # shortcut: 2023 fills gaps in 2024
gdp = e.set_index("country_name").gdp

Countries with GDP by year (non-null): {'1960': 152, '1961': 155, '1962': 157, '1963': 157, '1964': 157, '1965': 163, '1966': 164, '1967': 167, '1968': 168, '1969': 168, '1970': 190, '1971': 191, '1972': 191, '1973': 191, '1974': 191, '1975': 195, '1976': 195, '1977': 198, '1978': 197, '1979': 198, '1980': 208, '1981': 208, '1982': 209, '1983': 209, '1984': 211, '1985': 213, '1986': 213, '1987': 220, '1988': 223, '1989': 223, '1990': 240, '1991': 241, '1992': 242, '1993': 243, '1994': 244, '1995': 247, '1996': 247, '1997': 249, '1998': 250, '1999': 250, '2000': 252, '2001': 253, '2002': 257, '2003': 257, '2004': 257, '2005': 257, '2006': 258, '2007': 258, '2008': 259, '2009': 261, '2010': 261, '2011': 262, '2012': 260, '2013': 260, '2014': 261, '2015': 260, '2016': 259, '2017': 259, '2018': 259, '2019': 259, '2020': 258, '2021': 258, '2022': 257, '2023': 252, '2024': 248, '2025': 234}


In [53]:
#10. Country salary table (>=50 postings)
junior = g[g.seniority.isin(["entry", "intern"])]
c = g.groupby("country_name").salary_mid_usd.agg(n="count", median_all="median")
c["median_junior"] = junior.groupby("country_name").salary_mid_usd.median()
c = c[c.n >= 50].join(gdp)
c["gdp_trillion"] = c.gdp / 1e12
print("\n=== Country salary vs GDP ===")
print(c.sort_values("median_all", ascending=False).round(2).to_string())
 
d = c.dropna(subset=["gdp", "median_junior"])
print(f"\nSpearman, median_all vs log GDP:   r={stats.spearmanr(c.dropna(subset=['gdp']).median_all, c.dropna(subset=['gdp']).gdp)[0]:.2f}  (n={len(c.dropna(subset=['gdp']))})")
print(f"Spearman, median_junior vs GDP:   r={stats.spearmanr(d.median_junior, d.gdp)[0]:.2f}  (n={len(d)})")
# India is the big-GDP outlier: a large economy with very low pay -> shows total GDP is NOT a pay measure
print("India GDP rank among these countries:", int(c.gdp.rank(ascending=False)["India"]), "of", len(c.dropna(subset=["gdp"])))
ex = c.drop("India").dropna(subset=["gdp"])
print(f"Without India: median_all vs GDP r={stats.spearmanr(ex.median_all, ex.gdp)[0]:.2f} (n={len(ex)})")


=== Country salary vs GDP ===
                    n  median_all  median_junior           gdp  gdp_trillion
country_name                                                                
Switzerland       354    116867.5       111446.0  9.699198e+11          0.97
Ireland            71     66742.0        44944.0  6.091575e+11          0.61
United States   56906     60000.0        41600.0  2.929801e+13         29.30
Austria           360     55056.0        43820.0  5.347907e+11          0.53
Netherlands       883     47191.0        43483.0  1.213936e+12          1.21
United Kingdom    371     46053.0        36184.0  3.695540e+12          3.70
Canada          18617     44643.0        37143.0  2.270076e+12          2.27
Germany          2703     44638.0        53753.0  4.685593e+12          4.69
Belgium            78     41798.0        44494.5  6.709831e+11          0.67
Singapore        1319     35267.0        25191.0  5.728773e+11          0.57
France          10938     31551.0        3735

In [54]:
#11. Where is Rs 15 lakh in India? 
THRESH = 1_500_000
RATE = 86  # INR per USD implied by the global file
print(f"\n=== Rs 15 lakh = USD {THRESH/RATE:,.0f} ===")
print("Share of ALL Indian postings paying >= Rs 15 lakh: %.1f%%" % ((ind.salary_mid_inr >= THRESH).mean() * 100))
b = ind.groupby("experience_band").salary_mid_inr.agg(
    median="median", pct_above_15L=lambda s: (s >= THRESH).mean() * 100).round(1)
b = b.reindex(["0-1 yrs", "2-3 yrs", "4-6 yrs", "7-10 yrs", "11-15 yrs", "16+ yrs"])
print(b.to_string())
 
# 15 lakh vs median pay abroad
usd15 = THRESH / RATE
print(f"\nRs 15 lakh (${usd15:,.0f}) vs median pay (all seniorities, USD):")
for k in ["United States", "Canada", "Singapore", "France", "Germany", "India"]:
    m = g[g.country_name == k].salary_mid_usd.median()
    print(f"  {k:14s} median ${m:>9,.0f}  -> Rs15L is {usd15/m:.0%} of it")
print(f"Share of US postings paying less than ${usd15:,.0f}: {(g[g.country_name=='United States'].salary_mid_usd < usd15).mean()*100:.1f}%")


=== Rs 15 lakh = USD 17,442 ===
Share of ALL Indian postings paying >= Rs 15 lakh: 19.6%
                    median  pct_above_15L
experience_band                          
0-1 yrs           300000.0            1.2
2-3 yrs           525000.0           10.0
4-6 yrs           850000.0           30.8
7-10 yrs         1300000.0           46.4
11-15 yrs        1800000.0           69.8
16+ yrs          2125000.0           58.7

Rs 15 lakh ($17,442) vs median pay (all seniorities, USD):
  United States  median $   60,000  -> Rs15L is 29% of it
  Canada         median $   44,643  -> Rs15L is 39% of it
  Singapore      median $   35,267  -> Rs15L is 49% of it
  France         median $   31,551  -> Rs15L is 55% of it
  Germany        median $   44,638  -> Rs15L is 39% of it
  India          median $    6,977  -> Rs15L is 250% of it
Share of US postings paying less than $17,442: 0.0%


In [55]:
#11.Plot 
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
p = c.dropna(subset=["gdp"])
ax[0].scatter(p.gdp_trillion, p.median_all)
for k, r in p.iterrows():
    ax[0].annotate(k, (r.gdp_trillion, r.median_all), fontsize=8)
ax[0].set_xscale("log"); ax[0].set_xlabel("GDP 2024 (US$ trillion, log)"); ax[0].set_ylabel("Median posted salary (USD)")
ax[0].set_title("Country GDP vs median pay")
b.pct_above_15L.plot.bar(ax=ax[1], color="#55a868")
ax[1].set_title("India: % of postings paying >= Rs 15 lakh"); ax[1].set_ylabel("%")
plt.tight_layout(); plt.savefig(f"{OUT}/gdp_and_15lakh.png", dpi=130)
print("saved")

saved
